# Create CFREF Awards (Canada First Research Excellence Fund)

Creates Canada First Research Excellence Fund awards from the fund's own
competition-results pages at https://www.cfref-apogee.gc.ca/results-resultats/
(Inaugural Competition 1, Inaugural Competition 2, 2022 Competition). Each award
block gives the lead institution, award amount (CAD), initiative title, partner
institutions (2022 only), a summary and a full abstract page. **29 awards,
CAD 2.63B total, 100% title/amount/abstract/start date** (local run 2026-09-30).

CFREF awards are made to institutions, not to a named PI, so
`lead_investigator` carries only the affiliation (lead institution); given and
family names are NULL by design (the fund does not publish a person).

**Prerequisites:**
- Run `scripts/local/cfref_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/cfref/cfref_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** citing works write the CFREF grant number
(`CFREF-2015-00013`, `CFREF-2022-00042`; dominant form among the ~340
`crossref_work_funders` / `crossref_work.grants` shells for F4320326644). The
CFREF site does not print it; the script maps (competition, lead institution)
to the number using the NSERC Awards open data, which carries all 29 CFREF
awards as application ids `cfref-2014-0000N` (Competition 1),
`cfref-2015-000NN` (Competition 2) and `cfref-2022-000NN` (2022 Competition),
one per (competition, institution). Shipping that string lets these rows
collapse onto the existing citation shells in CreateAwards dedup.

**start_date** = the announcement date from the fund's own press-release index
(Competition 1: one release per institution, 2015-07-28..31; Competition 2:
2016-09-06; 2022 Competition: 2023-04-28). No end dates are published.

**Overlap note:** NSERC (`nserc_open_data`), SSHRC and CIHR open data also carry
their own yearly CFREF disbursement rows under their own funder ids
(funder_scheme 'Canada First Research Excellence Fund'). Those are the agencies'
tranches, not the CFREF award; different funder_id, so no id collision.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320326644`
- display_name: Canada First Research Excellence Fund
- ror_id / doi: from `openalex.funders.funders` (ror 01ktx4s83, doi 10.13039/501100010785)

**Priority:** `482` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cfref_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cfref/cfref_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.cfref_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.cfref_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cfref_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320326644 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320326644;

## Step 2: Create CFREF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cfref_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320326644  -- Canada First Research Excellence Fund
),

lead AS (
    -- Institution-only lead (CFREF names no PI).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        named_struct(
            'given_name', CAST(NULL AS STRING),
            'family_name', CAST(NULL AS STRING),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'CA',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_investigator
    FROM openalex.awards.cfref_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    COALESCE(NULLIF(TRIM(g.abstract), ''), NULLIF(TRIM(g.summary), '')) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    g.currency as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('CFREF ', g.competition) as funder_scheme,
    'cfref_competition_results' as provenance,
    TRY_TO_DATE(g.announcement_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.announcement_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    COALESCE(g.abstract_url, g.results_page_url) as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.cfref_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id must be CFREF-YYYY-NNNNN.
SELECT assert_true(COUNT_IF(NOT funder_award_id RLIKE '^CFREF-20(14|15|22)-[0-9]{5}$') = 0,
                   'funder_award_id not in CFREF-YYYY-NNNNN form') AS shape_ok
FROM openalex.awards.cfref_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cfref_competition_results' AND priority = 482;

-- Insert into openalex_awards_raw with priority.
-- Priority 482: direct-from-funder ingest of CFREF's own competition results.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 482 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    482 as priority
FROM openalex.awards.cfref_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_cfref_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.cfref_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funder_scheme, amount, currency, start_date,
       lead_investigator.affiliation.name AS institution, landing_page_url
FROM openalex.awards.cfref_awards ORDER BY funder_award_id;

In [ ]:
%sql
-- Section 6.7 coverage (expect 100% title/amount/description/start_date; no PI names by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(SUM(amount), 0) as total_cad,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.cfref_awards;

In [ ]:
%sql
-- Existing citation shells for F4320326644 that these awards upgrade.
SELECT c.provenance, COUNT(*) as shells, COUNT(t.id) as matched_by_cfref_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.cfref_awards t ON t.id = c.id
WHERE c.funder_id = 4320326644 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cfref_competition_results'
GROUP BY provenance, priority;